# Estudo de Caso 4.8 — Sistema Hidráulico para Elevação de Carga Pesada: Dimensionamento e Análise de NPSH

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 4 — Escoamento em Tubulações e Bombeamento |
| **Nível** | Pós-Graduação |
| **Tempo estimado** | 75 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | Solução direta com fator de atrito iterativo (Colebrook–White) e análise comparativa de cenários |
| **Bibliografia** | Gobbi (s.d.), Cap. 2; Lugon Junior et al. (2026); White, *Mecânica dos Fluidos*, Cap. 6 |

---

## 📋 Resumo

Este estudo de caso dimensiona um sistema hidráulico industrial para elevação de uma carga de 10 toneladas distribuída em dois cilindros de ação dupla. O fenômeno modelado é o escoamento de óleo hidráulico ISO VG 46 em regime permanente, com cálculo de pressões, vazões e perdas de carga pela equação de Darcy–Weisbach com fator de atrito obtido iterativamente pela equação de Colebrook–White. O resultado central é a potência do motor-bomba e, sobretudo, a **análise de NPSH** para prevenção de cavitação, comparando um projeto original inviável com um projeto corrigido.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. **Dimensionar** cilindros hidráulicos de alta pressão a partir da carga e da pressão de trabalho.
2. **Calcular** perdas de carga distribuídas e localizadas em linhas de pressão e sucção usando Colebrook–White.
3. **Determinar** a potência hidráulica e elétrica do conjunto motor-bomba considerando eficiências volumétrica e mecânica.
4. **Verificar** o NPSH disponível em relação ao NPSH requerido para prevenir cavitação.
5. **Comparar** cenários de projeto (original vs. corrigido) e propor ações corretivas fundamentadas.
6. **Analisar** a sensibilidade do NPSH à variação da velocidade de elevação e do diâmetro de sucção.
7. **Interpretar** o impacto da velocidade de escoamento na viabilidade de sistemas hidráulicos de sucção.

> 📖 **Referência:** Gobbi (s.d.), Cap. 2; Lugon Junior et al. (2026).

## ⚙️ Seção 0: Configuração Inicial

Importação das bibliotecas científicas e definição do estilo de gráficos conforme o guia de padronização.

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
import time

# Estilo padronizado de gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("=" * 60)
print("📦 CONFIGURAÇÃO DO AMBIENTE")
print("=" * 60)
print("  • numpy      : operações vetorizadas")
print("  • matplotlib : visualização padronizada")
print("  • pandas     : tabelas comparativas")
print("✅ Ambiente configurado com sucesso!")

## 🌊 Seção 1: Parâmetros do Sistema Hidráulico

Um sistema hidráulico industrial deve elevar uma carga de **10.000 kg** (98,1 kN) distribuída simetricamente entre **dois cilindros de ação dupla**.

| Parâmetro | Símbolo | Valor | Unidade |
|:---|:---:|:---:|:---:|
| Massa da carga | $m$ | 10.000 | kg |
| Número de cilindros | $n$ | 2 | — |
| Diâmetro do êmbolo | $D_e$ | 80 | mm |
| Diâmetro da haste | $D_h$ | 45 | mm |
| Curso do cilindro | $s$ | 500 | mm |
| Velocidade de elevação | $v$ | 0,15 | m/s |
| Rugosidade da mangueira/tubo | $\varepsilon$ | 0,01 | mm |
| Pressão atmosférica | $p_{atm}$ | 1,013 | bar |
| Pressão de vapor do óleo (40 °C) | $p_{vap}$ | 0,15 | bar |

**Propriedades do óleo ISO VG 46 (40 °C)**

| Propriedade | Símbolo | Valor | Unidade |
|:---|:---:|:---:|:---:|
| Massa específica | $\rho$ | 870 | kg/m³ |
| Viscosidade cinemática | $\nu$ | 46 × 10⁻⁶ | m²/s |

**Eficiências da bomba**

| Parâmetro | Símbolo | Valor |
|:---|:---:|:---:|
| Rendimento volumétrico | $\eta_v$ | 0,92 |
| Rendimento mecânico | $\eta_m$ | 0,88 |

**Tubulações e acessórios**

| Trecho | Diâmetro | Comprimento | Acessórios |
|:---|:---:|:---:|:---|
| Pressão (mangueira) | DN 16 | 4,5 m | válvula direcional, retenção, 4 curvas |
| Sucção (original) | DN 20 | 2,0 m | filtro de alta perda, 2 curvas |
| Sucção (corrigida) | DN 25 | 2,0 m | filtro de baixa perda, 2 curvas |

In [ ]:
# ============================================================
# 1. PARÂMETROS DO SISTEMA E DO FLUIDO
# ============================================================
# --- Carga e geometria ---
m_carga   = 10000.0     # Massa da carga [kg]
g         = 9.81        # Aceleração da gravidade [m/s²]
F_total   = m_carga * g # Força peso total [N]
n_cil     = 2           # Número de cilindros [-]
D_embolo  = 0.080       # Diâmetro do êmbolo [m]
D_haste   = 0.045       # Diâmetro da haste [m]
curso     = 0.500       # Curso do cilindro [m]
v_elev    = 0.15        # Velocidade de elevação [m/s]

# --- Fluido (ISO VG 46 a 40 °C) ---
rho   = 870.0           # Massa específica [kg/m³]
nu    = 46e-6           # Viscosidade cinemática [m²/s]
gamma = rho * g         # Peso específico [N/m³]

# --- Tubulação de pressão (mangueira) ---
D_mang   = 0.016        # Diâmetro interno [m] (DN 16)
L_mang   = 4.5          # Comprimento [m]
epsilon  = 0.01e-3      # Rugosidade absoluta [m]

# --- Tubulação de sucção ---
L_suc    = 2.0          # Comprimento [m]
z_bomba  = 0.30         # Altura da bomba acima do nível do óleo [m]
z_res    = 0.0          # Cota do nível do óleo [m]
K_curva  = 0.9          # Coeficiente de perda por curva [-]

# --- Perdas de referência dos acessórios ---
P_valv_ref  = 3.0       # Perda na válvula direcional a 40 L/min [bar]
Q_valv_ref  = 40.0      # Vazão de referência da válvula [L/min]
P_ret_ref   = 0.5       # Perda na válvula de retenção a 100 L/min [bar]
Q_ret_ref   = 100.0     # Vazão de referência da retenção [L/min]
P_filtro_orig = 0.8     # Perda no filtro original a 100 L/min [bar]
P_filtro_corr = 0.2     # Perda no filtro corrigido a 100 L/min [bar]
Q_filtro_ref  = 100.0   # Vazão de referência do filtro [L/min]

# --- Eficiências e pressões ---
eta_v   = 0.92          # Rendimento volumétrico [-]
eta_m   = 0.88          # Rendimento mecânico [-]
P_atm   = 1.013e5       # Pressão atmosférica [Pa]
P_vap   = 0.15e5        # Pressão de vapor do óleo [Pa]
NPSH_req = 3.5          # NPSH requerido pela bomba [m]

print("=" * 60)
print("🌊 PARÂMETROS DO SISTEMA")
print("=" * 60)
print(f"  • Carga total      : {F_total/1000:.1f} kN ({m_carga/1000:.0f} t)")
print(f"  • Velocidade elevação: {v_elev} m/s")
print(f"  • Óleo ISO VG 46   : ρ={rho} kg/m³, ν={nu*1e6:.0f} cSt")
print(f"  • Bomba acima do nível: z = {z_bomba} m")

## 🔬 Seção 2: Verificação / Validação

Antes do dimensionamento, validamos a consistência física dos parâmetros:

1. **Faixa de pressão de trabalho:** a pressão teórica deve ser inferior à pressão máxima admissível de componentes industriais típicos (~210 bar).
2. **Faixa de velocidade na linha de pressão:** recomendação de 3–6 m/s para óleo hidráulico.
3. **Consistência da continuidade:** $Q = v \cdot A$.

> 💡 **Observação:** a verificação prévia evita propagar dados inconsistentes e confirma a aplicabilidade das correlações e faixas de projeto.

In [ ]:
# ============================================================
# 2. VERIFICAÇÃO PRÉVIA DOS PARÂMETROS
# ============================================================
A_emb    = np.pi * D_embolo**2 / 4        # Área do êmbolo [m²]
F_cil    = F_total / n_cil                # Força por cilindro [N]
p_teor   = F_cil / A_emb / 1e5            # Pressão teórica [bar]
Q_cil    = A_emb * v_elev                 # Vazão por cilindro [m³/s]
Q_bomba  = n_cil * Q_cil * 1.08           # Vazão da bomba com 8% de vazamento [m³/s]
A_mang   = np.pi * D_mang**2 / 4          # Área da mangueira [m²]
u_mang   = Q_bomba / A_mang               # Velocidade na mangueira [m/s]

print("=" * 60)
print("🔬 VERIFICAÇÃO PRÉVIA")
print("=" * 60)
print(f"📏 Área do êmbolo      : A  = {A_emb:.4e} m²")
print(f"📏 Pressão teórica     : p  = {p_teor:.2f} bar")
print(f"📏 Vazão da bomba      : Q  = {Q_bomba*60000:.1f} L/min")
print(f"📏 Velocidade mangueira: u  = {u_mang:.2f} m/s")

# Verificação 1: pressão teórica abaixo de 210 bar
assert p_teor < 210.0, f"⚠️ Pressão teórica {p_teor:.1f} bar excede 210 bar"
print("✅ Verificação 1: pressão teórica dentro do limite (210 bar)")

# Verificação 2: continuidade Q = v*A
Q_recalc = v_elev * A_emb
erro_cont = abs(Q_recalc - Q_cil) / Q_cil * 100
assert erro_cont < 1e-9, "❌ Erro na continuidade"
print("✅ Verificação 2: equação da continuidade satisfeita")

# Verificação 3: velocidade na linha de pressão
if 3.0 <= u_mang <= 6.0:
    print(f"✅ Verificação 3: velocidade na faixa recomendada (3–6 m/s)")
else:
    print(f"⚠️ Verificação 3: velocidade {u_mang:.2f} m/s fora da faixa recomendada (3–6 m/s)")
    print("   → Em projeto real, considerar duas mangueiras em paralelo ou DN maior")

print("\n✅ Verificações prévias concluídas!")

## 📐 Seção 3: Formulação Matemática

### 3.1 Pressão nos cilindros

A pressão teórica necessária para elevar a carga é:

$$p_{teor} = \frac{F_{cil}}{A_{e}}$$

com margem de 15% para atrito das vedações:

$$p_{trab} = 1{,}15 \, p_{teor}$$

### 3.2 Fator de atrito (Colebrook–White)

Equação implícita resolvida iterativamente:

$$\frac{1}{\sqrt{f}} = -2\,\log_{10}\left(\frac{\varepsilon/D}{3{,}7} + \frac{2{,}51}{Re\,\sqrt{f}}\right)$$

Para regime laminar ($Re < 2300$), usa-se $f = 64/Re$.

### 3.3 Perdas de carga

Perda distribuída (Darcy–Weisbach) e perda localizada:

$$\Delta p_f = f\,\frac{L}{D}\,\frac{\rho u^2}{2}, \qquad \Delta p_s = K\,\frac{\rho u^2}{2}$$

Perdas em acessórios são ajustadas quadraticamente pela vazão real:

$$\Delta p = \Delta p_{ref}\left(\frac{Q}{Q_{ref}}\right)^2$$

### 3.4 NPSH disponível

$$NPSH_d = \frac{p_{atm} - p_{vap}}{\gamma} + z_{res} - z_{bomba} - h_{f,suc} - \frac{u_{suc}^2}{2g}$$

onde $h_{f,suc}$ inclui perdas distribuídas, localizadas e no filtro de sucção.

> 📖 **Referência:** Gobbi (s.d.), Cap. 2; White, Cap. 6.

## ⚡ Seção 4: Estabilidade / Critério Numérico

> 💡 **Observação:** trata-se de **escoamento permanente** com correlações empíricas; portanto, **não se aplica a condição CFL**. Mantém-se esta seção com o critério físico de regime.

O critério de regime é dado pelo **número de Reynolds**:

$$Re = \frac{u\,D}{\nu}$$

| Faixa de $Re$ | Regime | Fator de atrito |
|:---:|:---|:---|
| $Re < 2300$ | Laminar | $f = 64/Re$ |
| $2300 \le Re \le 4000$ | Transição | Interpolação |
| $Re > 4000$ | Turbulento | Colebrook–White |

A **iteração de Colebrook–White** converge quando $|f_{novo} - f_{antigo}| < 10^{-6}$, tipicamente em menos de 20 iterações.

## 🌪️ Seção 5: Forçamento / Condição de Entrada

O **forçamento** do sistema é a **velocidade de elevação** $v = 0{,}15$ m/s imposta pela demanda do processo. As **condições de contorno** são:

- **Sucção:** pressão atmosférica no reservatório de óleo ($p = p_{atm}$), nível na cota $z_{res} = 0$.
- **Pressão:** a bomba fornece a pressão necessária para vencer a carga e as perdas, alimentando os cilindros.

A vazão é determinada pela continuidade $Q = v \cdot A_{e}$, multiplicada pelo fator de vazamento (8%).

## 🧮 Seção 6: Implementação Numérica

As funções abaixo encapsulam os cálculos hidráulicos, com destaque para a **iteração de Colebrook–White** e a **análise de NPSH** para os dois cenários de sucção.

In [ ]:
# ============================================================
# 6.1 FUNÇÕES HIDRÁULICAS
# ============================================================
def fator_atrito_colebrook(Re, eps_D):
    """
    Calcula o fator de atrito de Darcy pela equação de Colebrook-White.

    Parameters
    ----------
    Re : float
        Número de Reynolds [-].
    eps_D : float
        Rugosidade relativa (epsilon/D) [-].

    Returns
    -------
    float
        Fator de atrito de Darcy f [-].

    Notes
    -----
    Para Re < 2300 usa f = 64/Re (laminar). Para regime turbulento,
    resolve iterativamente a equação de Colebrook-White.
    """
    if Re < 2300:
        return 64.0 / Re
    f = 0.02  # chute inicial
    for _ in range(20):
        f_novo = (-2.0 * np.log10(eps_D / 3.7 + 2.51 / (Re * np.sqrt(f))))**(-2)
        if abs(f - f_novo) < 1e-6:
            break
        f = f_novo
    return f


def perda_darcy_Pa(f, L, D, rho, u):
    """
    Calcula a perda de pressão distribuída (Darcy-Weisbach) em Pa.

    Parameters
    ----------
    f : float
        Fator de atrito de Darcy [-].
    L : float
        Comprimento da tubulação [m].
    D : float
        Diâmetro interno [m].
    rho : float
        Massa específica [kg/m³].
    u : float
        Velocidade média [m/s].

    Returns
    -------
    float
        Perda de pressão [Pa].
    """
    return f * (L / D) * (rho * u**2 / 2)


def calcular_NPSH(D_suc, dP_filtro_ref_bar, Q_ref_filtro_Lmin, Q_bomba_m3s):
    """
    Calcula o NPSH disponível para um dado diâmetro de sucção e filtro.

    Parameters
    ----------
    D_suc : float
        Diâmetro interno da tubulação de sucção [m].
    dP_filtro_ref_bar : float
        Perda de pressão de referência no filtro [bar].
    Q_ref_filtro_Lmin : float
        Vazão de referência do filtro [L/min].
    Q_bomba_m3s : float
        Vazão da bomba [m³/s].

    Returns
    -------
    dict
        Dicionário com velocidade, Reynolds, perdas e NPSH disponível.
    """
    Q_Lmin = Q_bomba_m3s * 60000
    A_suc = np.pi * D_suc**2 / 4
    u_suc = Q_bomba_m3s / A_suc
    Re_suc = u_suc * D_suc / nu
    eps_D_suc = epsilon / D_suc

    f_suc = fator_atrito_colebrook(Re_suc, eps_D_suc)

    # Perdas em metros de coluna de óleo (m.c.o.)
    hf_dist = f_suc * (L_suc / D_suc) * (u_suc**2 / (2 * g))
    hf_curvas = (2 * K_curva) * (u_suc**2 / (2 * g))

    # Perda no filtro ajustada pela vazão e convertida para metros
    dP_filtro_Pa_real = (dP_filtro_ref_bar * 1e5) * (Q_Lmin / Q_ref_filtro_Lmin)**2
    hf_filtro = dP_filtro_Pa_real / (rho * g)

    hf_total_suc = hf_dist + hf_curvas + hf_filtro

    # NPSH disponível
    termo_pressao = (P_atm - P_vap) / (rho * g)
    NPSH_d = termo_pressao + z_res - z_bomba - hf_total_suc - u_suc**2 / (2 * g)

    return {
        'u_suc': u_suc, 'Re_suc': Re_suc, 'f_suc': f_suc,
        'hf_dist': hf_dist, 'hf_curvas': hf_curvas, 'hf_filtro': hf_filtro,
        'hf_total': hf_total_suc, 'NPSH_disp': NPSH_d
    }


print("✅ Funções hidráulicas definidas com sucesso!")

In [ ]:
# ============================================================
# 6.2 CÁLCULO PRINCIPAL — PRESSÃO, VAZÃO E POTÊNCIA
# ============================================================
t0 = time.time()

# Passo 1: pressão de trabalho
A_embolo   = np.pi * D_embolo**2 / 4          # Área do êmbolo [m²]
F_cilindro = F_total / n_cil                  # Força por cilindro [N]
p_teorica_Pa  = F_cilindro / A_embolo         # Pressão teórica [Pa]
p_teorica_bar = p_teorica_Pa / 1e5            # Pressão teórica [bar]
p_trabalho_bar = p_teorica_bar * 1.15         # Com margem de 15% [bar]
p_alivio_bar   = p_trabalho_bar * 1.5         # Válvula de alívio (FS 1,5) [bar]

# Passo 2: vazão da bomba
Q_cil_m3s   = A_embolo * v_elev               # Vazão por cilindro [m³/s]
Q_cil_Lmin  = Q_cil_m3s * 60000               # Vazão por cilindro [L/min]
fator_vaz   = 1.08                            # Fator de vazamento [-]
Q_bomba_Lmin = n_cil * Q_cil_Lmin * fator_vaz # Vazão da bomba [L/min]
Q_bomba_m3s  = Q_bomba_Lmin / 60000           # Vazão da bomba [m³/s]

# Passo 3: perdas na linha de pressão
A_mang   = np.pi * D_mang**2 / 4              # Área da mangueira [m²]
u_mang   = Q_bomba_m3s / A_mang               # Velocidade [m/s]
Re_mang  = u_mang * D_mang / nu               # Reynolds [-]
eps_D_mang = epsilon / D_mang                 # Rugosidade relativa [-]
f_mang   = fator_atrito_colebrook(Re_mang, eps_D_mang)

dP_dist_Pa   = perda_darcy_Pa(f_mang, L_mang, D_mang, rho, u_mang)
dP_valv_bar  = P_valv_ref * (Q_bomba_Lmin / Q_valv_ref)**2
dP_ret_bar   = P_ret_ref * (Q_bomba_Lmin / Q_ret_ref)**2
dP_curvas_Pa = (4 * K_curva) * (rho * u_mang**2 / 2)
dP_curvas_bar = dP_curvas_Pa / 1e5
dP_total_pressao_bar = (dP_dist_Pa / 1e5) + dP_valv_bar + dP_ret_bar + dP_curvas_bar

# Passo 4: potência
p_descarga_bar = p_trabalho_bar + dP_total_pressao_bar
p_descarga_Pa  = p_descarga_bar * 1e5
P_hid_W  = (p_descarga_Pa * Q_bomba_m3s) / eta_v   # Potência hidráulica [W]
P_elet_W = P_hid_W / eta_m                          # Potência elétrica [W]
P_elet_kW = P_elet_W / 1000                         # Potência elétrica [kW]

tempo_calc = time.time() - t0

print("=" * 60)
print(" RESULTADOS DO DIMENSIONAMENTO")
print("=" * 60)
print(f"\n🔧 Cilindros:")
print(f"  • Área do êmbolo       : {A_embolo:.4e} m²")
print(f"  • Força por cilindro   : {F_cilindro/1000:.1f} kN")
print(f"  • Pressão teórica      : {p_teorica_bar:.1f} bar")
print(f"  • Pressão de trabalho  : {p_trabalho_bar:.1f} bar")
print(f"  • Pressão de alívio    : {p_alivio_bar:.1f} bar")
print(f"\n💧 Vazão:")
print(f"  • Vazão por cilindro   : {Q_cil_Lmin:.1f} L/min")
print(f"  • Vazão da bomba       : {Q_bomba_Lmin:.1f} L/min")
print(f"\n📉 Perdas na linha de pressão:")
print(f"  • Velocidade mangueira : {u_mang:.2f} m/s (Re={Re_mang:.0f})")
print(f"  • Fator de atrito      : {f_mang:.4f}")
print(f"  • Perda distribuída    : {dP_dist_Pa/1e5:.2f} bar")
print(f"  • Válvula direcional   : {dP_valv_bar:.2f} bar")
print(f"  • Válvula de retenção  : {dP_ret_bar:.2f} bar")
print(f"  • 4 curvas             : {dP_curvas_bar:.2f} bar")
print(f"  • Perda TOTAL          : {dP_total_pressao_bar:.2f} bar")
print(f"\n🔋 Potência:")
print(f"  • Pressão de descarga  : {p_descarga_bar:.1f} bar")
print(f"  • Potência hidráulica  : {P_hid_W/1000:.1f} kW")
print(f"  • Potência elétrica    : {P_elet_kW:.1f} kW")
print(f"\n⏱️ Tempo de cálculo: {tempo_calc*1000:.2f} ms")

In [ ]:
# ============================================================
# 6.3 ANÁLISE DE NPSH — CENÁRIOS ORIGINAL E CORRIGIDO
# ============================================================
# Cenário original: DN 20 com filtro de alta perda (0,8 bar)
cen_original = calcular_NPSH(D_suc=0.020, dP_filtro_ref_bar=P_filtro_orig,
                             Q_ref_filtro_Lmin=Q_filtro_ref, Q_bomba_m3s=Q_bomba_m3s)

# Cenário corrigido: DN 25 com filtro de baixa perda (0,2 bar)
cen_corrigido = calcular_NPSH(D_suc=0.025, dP_filtro_ref_bar=P_filtro_corr,
                              Q_ref_filtro_Lmin=Q_filtro_ref, Q_bomba_m3s=Q_bomba_m3s)

print("=" * 60)
print("⚠️ ANÁLISE DE NPSH — SUCÇÃO")
print("=" * 60)
print(f"{'Parâmetro':<24} | {'Original (DN20)':<15} | {'Corrigido (DN25)':<15}")
print("-" * 62)
print(f"Velocidade sucção        | {cen_original['u_suc']:>6.2f} m/s    | {cen_corrigido['u_suc']:>6.2f} m/s")
print(f"Reynolds                 | {cen_original['Re_suc']:>9.0f}     | {cen_corrigido['Re_suc']:>9.0f}")
print(f"Perda no filtro          | {cen_original['hf_filtro']:>6.2f} m      | {cen_corrigido['hf_filtro']:>6.2f} m")
print(f"Perda total sucção       | {cen_original['hf_total']:>6.2f} m      | {cen_corrigido['hf_total']:>6.2f} m")
print(f"NPSH disponível          | {cen_original['NPSH_disp']:>6.2f} m      | {cen_corrigido['NPSH_disp']:>6.2f} m")

margem_orig = cen_original['NPSH_disp'] - NPSH_req
margem_corr = cen_corrigido['NPSH_disp'] - NPSH_req

print(f"\nNPSH requerido pela bomba: {NPSH_req:.2f} m")
if margem_orig < 0:
    print(f"❌ Cenário ORIGINAL: NPSH_d = {cen_original['NPSH_disp']:.2f} m → RISCO DE CAVITAÇÃO")
else:
    print(f"✅ Cenário ORIGINAL: NPSH_d = {cen_original['NPSH_disp']:.2f} m → viável")

if margem_corr > 0:
    print(f"✅ Cenário CORRIGIDO: NPSH_d = {cen_corrigido['NPSH_disp']:.2f} m → VIÁVEL (margem {margem_corr:.2f} m)")
else:
    print(f"❌ Cenário CORRIGIDO: NPSH_d = {cen_corrigido['NPSH_disp']:.2f} m → ainda inviável")

## 📊 Seção 7: Visualização

In [ ]:
# ============================================================
# 7.1 COMPOSIÇÃO DAS PERDAS NA SUCÇÃO
# ============================================================
labels = ['Perda distribuída', 'Perda curvas', 'Perda filtro']
perdas_orig = [cen_original['hf_dist'], cen_original['hf_curvas'], cen_original['hf_filtro']]
perdas_corr = [cen_corrigido['hf_dist'], cen_corrigido['hf_curvas'], cen_corrigido['hf_filtro']]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, perdas_orig, width, label='Original (DN 20 + filtro 0,8 bar)', color='#C44E52')
rects2 = ax.bar(x + width/2, perdas_corr, width, label='Corrigido (DN 25 + filtro 0,2 bar)', color='#55A868')

ax.set_ylabel('Perda de carga [m.c.o.]', fontsize=12)
ax.set_title('Composição das Perdas de Carga na Linha de Sucção', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(fontsize=11)
ax.grid(True, axis='y', linestyle='--', alpha=0.3)

# Rótulos sobre as barras
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=10)

autolabel(rects1)
autolabel(rects2)
plt.tight_layout()
plt.show()

> 💡 **Observação:** o filtro de sucção é o componente dominante nas perdas do cenário original. A substituição por um filtro de baixa perda, combinada com o aumento do diâmetro da tubulação, reduz drasticamente a perda total e torna o NPSH disponível positivo.

In [ ]:
# ============================================================
# 7.2 NPSH DISPONÍVEL VS. REQUERIDO
# ============================================================
cenarios = ['Original\n(DN 20)', 'Corrigido\n(DN 25)']
npsh_valores = [cen_original['NPSH_disp'], cen_corrigido['NPSH_disp']]
cores = ['#C44E52' if v < NPSH_req else '#55A868' for v in npsh_valores]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(cenarios, npsh_valores, color=cores, edgecolor='black', linewidth=1.2)
ax.axhline(y=NPSH_req, color='blue', linestyle='--', linewidth=2,
           label=f'NPSH requerido ({NPSH_req} m)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.set_ylabel('NPSH [m]', fontsize=12)
ax.set_title('NPSH Disponível vs. Requerido', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, axis='y', linestyle='--', alpha=0.3)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.2, f'{yval:.2f} m',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

> 💡 **Observação:** o cenário original apresenta NPSH negativo, indicando cavitação certa. O cenário corrigido oferece margem confortável acima do NPSH requerido, garantindo operação segura.

## 📈 Seção 8: Análise Quantitativa

Consolidação dos resultados em tabela comparativa e métricas de desempenho.

In [ ]:
# ============================================================
# 8.1 TABELA DE RESULTADOS
# ============================================================
resultados = pd.DataFrame({
    'Parâmetro': ['Pressão de trabalho', 'Pressão de alívio', 'Vazão da bomba',
                  'Velocidade na mangueira', 'Perda total na pressão',
                  'Pressão de descarga', 'Potência elétrica',
                  'NPSH disponível (original)', 'NPSH disponível (corrigido)',
                  'Margem NPSH (corrigido)'],
    'Valor': [f'{p_trabalho_bar:.1f} bar', f'{p_alivio_bar:.1f} bar',
              f'{Q_bomba_Lmin:.1f} L/min', f'{u_mang:.2f} m/s',
              f'{dP_total_pressao_bar:.2f} bar', f'{p_descarga_bar:.1f} bar',
              f'{P_elet_kW:.1f} kW (motor ~{np.ceil(P_elet_kW/5)*5:.0f} kW)',
              f'{cen_original["NPSH_disp"]:.2f} m', f'{cen_corrigido["NPSH_disp"]:.2f} m',
              f'{margem_corr:.2f} m']
})

display(Markdown(resultados.to_markdown(index=False)))

print("\n" + "=" * 60)
print("📈 MÉTRICAS DE DESEMPENHO")
print("=" * 60)
print(f"  • Percentual de perdas sobre p_trab : {dP_total_pressao_bar/p_trabalho_bar*100:.1f}%")
print(f"  • Tempo total de cálculo            : {tempo_calc*1000:.2f} ms")
print(f"\n✅ Análise quantitativa concluída!")

## 🔍 Seção 9: Sensibilidade / Convergência

Analisamos como o **NPSH disponível** varia com a **velocidade de elevação** (que determina a vazão) e com o **diâmetro de sucção**, para os dois cenários de filtro. Todo o cálculo é **vetorizado**.

In [ ]:
# ============================================================
# 9.1 SENSIBILIDADE DO NPSH À VELOCIDADE DE ELEVAÇÃO
# ============================================================
v_range = np.linspace(0.05, 0.30, 100)     # Velocidades de 0,05 a 0,30 m/s
npsh_orig = np.zeros(len(v_range))
npsh_corr = np.zeros(len(v_range))

for i, v in enumerate(v_range):
    Q_i = n_cil * A_embolo * v * fator_vaz   # Vazão da bomba [m³/s]
    c_o = calcular_NPSH(0.020, P_filtro_orig, Q_filtro_ref, Q_i)
    c_c = calcular_NPSH(0.025, P_filtro_corr, Q_filtro_ref, Q_i)
    npsh_orig[i] = c_o['NPSH_disp']
    npsh_corr[i] = c_c['NPSH_disp']

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(v_range, npsh_orig, 'r-', linewidth=2, label='Original (DN 20 + filtro 0,8 bar)')
ax.plot(v_range, npsh_corr, 'g-', linewidth=2, label='Corrigido (DN 25 + filtro 0,2 bar)')
ax.axhline(y=NPSH_req, color='blue', linestyle='--', linewidth=2, label=f'NPSH requerido ({NPSH_req} m)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.axvline(x=v_elev, color='gray', linestyle=':', linewidth=2, label=f'Projeto ({v_elev} m/s)')
ax.set_xlabel('Velocidade de elevação, v [m/s]', fontsize=12)
ax.set_ylabel('NPSH disponível [m]', fontsize=12)
ax.set_title('Sensibilidade do NPSH à Velocidade de Elevação', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Insight: aumentar a velocidade de elevação reduz o NPSH disponível.")
print("   O cenário corrigido mantém margem segura em toda a faixa de operação.")

> 💡 **Observação:** a sensibilidade confirma que o NPSH decai com o aumento da vazão (e, portanto, da velocidade). O cenário original torna-se inviável já em velocidades moderadas, enquanto o cenário corrigido permanece seguro mesmo em velocidades mais altas.

## 💡 Seção 10: Discussão

### Interpretação física dos resultados

1. **Domínio do filtro na sucção:** no cenário original, o filtro de alta perda é o componente que mais compromete o NPSH, evidenciando que um acessório aparentemente simples pode inviabilizar todo o sistema.
2. **Velocidade excessiva na mangueira:** a velocidade na linha de pressão (~8 m/s) está acima da faixa recomendada (3–6 m/s), indicando que, em projeto real, se usariam duas mangueiras em paralelo ou um DN maior.
3. **Margem de segurança do NPSH:** o cenário corrigido oferece margem confortável, eliminando o risco de cavitação e garantindo a vida útil da bomba.

### Limitações do modelo

- **Escoamento permanente idealizado:** não considera transientes de partida/parada nem golpes de pressão.
- **Eficiências constantes:** admite-se $\eta_v$ e $\eta_m$ fixos, quando na prática variam com o ponto de operação.
- **Perdas de referência dos acessórios:** os valores de perda dos filtros e válvulas são de catálogo e podem variar com o desgaste.
- **Geometria simplificada:** não considera o efeito da haste na área efetiva do cilindro em retorno.

### Aplicações práticas

- **Seleção de bombas industriais:** a pressão de descarga e a vazão permitem escolher a bomba em catálogo, aplicando margem de segurança.
- **Prevenção de cavitação:** a análise de NPSH fundamenta o dimensionamento da linha de sucção e a escolha do filtro.

### Recomendações de engenharia

1. Adotar o **cenário corrigido** (DN 25 + filtro de baixa perda) para a linha de sucção.
2. Aumentar o diâmetro da linha de pressão ou usar **duas mangueiras em paralelo** para reduzir a velocidade.
3. Instalar **válvula de alívio** ajustada em ~170 bar para proteger o sistema.
4. Para sincronismo dos dois cilindros sob cargas assimétricas, usar **divisor de fluxo** ou válvulas de controle de vazão compensadas.

### Conexão com a teoria

O caso aplica diretamente a conservação de energia (equação de Bernoulli estendida) e os conceitos de perda de carga do Capítulo 4, conectando a equação de Colebrook–White e a análise de NPSH ao dimensionamento real de sistemas hidráulicos industriais.

> 📖 **Referência:** Gobbi (s.d.), Cap. 2; Lugon Junior et al. (2026).

---

## 🔄 Seção 11: Navegação

- [📚 Voltar ao Capítulo 4](../notebooks/04_Escoamento_Tubulacoes.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">
Estudo de Caso 4.8 — EC_04_08_Elevador_Carga_Hidraulico.ipynb<br>
Parte da coleção <i>"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"</i>
</div>

In [ ]:
# ============================================================
# ENCERRAMENTO
# ============================================================
print("=" * 60)
print("✅ ESTUDO DE CASO 4.8 CONCLUÍDO COM SUCESSO!")
print("=" * 60)
print("\n🎓 Você completou o dimensionamento do sistema hidráulico.")
print("   Resultados principais:")
print(f"   • Pressão de trabalho : {p_trabalho_bar:.1f} bar")
print(f"   • Potência elétrica   : {P_elet_kW:.1f} kW")
print(f"   • NPSH original       : {cen_original['NPSH_disp']:.2f} m (inviável)")
print(f"   • NPSH corrigido      : {cen_corrigido['NPSH_disp']:.2f} m (viável)")
print("\n💡 Próximo passo: explore os estudos de caso do Capítulo 5 (Hidrodinâmica de Canais).")